# Interactive Weather Comparison

This notebook compares weather patterns across stations within one or more zones. Use the controls below to choose zones, a weather parameter, and a year, then inspect the Plotly charts for each selected zone.

In [1]:
from functools import lru_cache
from pathlib import Path
import re

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

pd.options.mode.copy_on_write = True

BASE_DIR = Path.cwd()
WEATHER_DIR = BASE_DIR / "weather_by_zone"

if not WEATHER_DIR.exists():
    WEATHER_DIR = BASE_DIR / "Notebooks" / "Weather" / "weather_by_zone"

if not WEATHER_DIR.exists():
    raise FileNotFoundError(f"Could not locate weather files under {BASE_DIR}.")

FILE_PATTERN = "weather_*.csv"


In [2]:
def parse_weather_filename(path: Path) -> dict:
    match = re.match(r"weather_(?P<zone>[^_]+)_(?P<location>.+)_(?P<station_id>\d+)\.csv$", path.name)
    if not match:
        raise ValueError(f"Unexpected weather filename: {path.name}")
    return match.groupdict()


def canonicalize_parameter_name(column_name: str) -> str:
    cleaned = re.sub(r"\s*\([^)]*\)", "", column_name)
    cleaned = re.sub(r"[^0-9a-zA-Z]+", "_", cleaned)
    return cleaned.strip("_").lower()


def list_weather_files() -> list[Path]:
    return sorted(WEATHER_DIR.glob(FILE_PATTERN))


def get_available_zones() -> list[str]:
    return sorted({parse_weather_filename(path)["zone"] for path in list_weather_files()})


@lru_cache(maxsize=None)
def get_parameter_map() -> dict[str, str]:
    sample_file = list_weather_files()[0]
    sample_df = pd.read_csv(sample_file, nrows=5)
    excluded_columns = {"location_id", "time", "zone"}
    parameter_map = {}
    for column in sample_df.columns:
        if column in excluded_columns:
            continue
        alias = canonicalize_parameter_name(column)
        parameter_map[alias] = column
    return dict(sorted(parameter_map.items()))


def get_display_name(parameter_alias: str) -> str:
    original = get_parameter_map()[parameter_alias]
    return f"{parameter_alias}  ->  {original}"


@lru_cache(maxsize=None)
def load_zone_weather(zone: str) -> pd.DataFrame:
    zone_files = sorted(WEATHER_DIR.glob(f"weather_{zone}_*.csv"))
    if not zone_files:
        raise FileNotFoundError(f"No files found for zone {zone}.")

    frames = []
    for path in zone_files:
        parts = parse_weather_filename(path)
        frame = pd.read_csv(path)
        frame["time"] = pd.to_datetime(frame["time"], errors="coerce")
        frame = frame.dropna(subset=["time"]).copy()
        frame["zone_code"] = parts["zone"]
        frame["station_name"] = parts["location"].replace("_", " ").title()
        frame["station_id"] = parts["station_id"]
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True)
    combined["year"] = combined["time"].dt.year
    return combined.sort_values(["station_name", "time"]).reset_index(drop=True)


def filter_by_year(df: pd.DataFrame, year: int) -> pd.DataFrame:
    return df.loc[df["year"] == year].copy()


def get_zone_parameter_frame(zone: str, parameter_alias: str, year: int) -> pd.DataFrame:
    parameter_column = get_parameter_map()[parameter_alias]
    zone_df = load_zone_weather(zone)
    filtered = filter_by_year(zone_df, year)
    if parameter_column not in filtered.columns:
        raise KeyError(f"Parameter '{parameter_alias}' is not available in zone {zone}.")

    filtered[parameter_alias] = pd.to_numeric(filtered[parameter_column], errors="coerce")
    return filtered.dropna(subset=[parameter_alias]).copy()


@lru_cache(maxsize=None)
def get_available_years() -> list[int]:
    years = set()
    for zone in get_available_zones():
        years.update(load_zone_weather(zone)["year"].dropna().astype(int).unique().tolist())
    return sorted(years)


In [3]:
def plot_weather_parameter(data: pd.DataFrame, parameter_alias: str, zone: str) -> go.Figure:
    parameter_column = get_parameter_map()[parameter_alias]
    figure = go.Figure()

    for station_name, station_df in data.groupby("station_name", sort=True):
        figure.add_trace(
            go.Scatter(
                x=station_df["time"],
                y=station_df[parameter_alias],
                mode="lines",
                name=station_name,
                hovertemplate=(
                    "Zone: %{meta[0]}<br>"
                    "Station: %{meta[1]}<br>"
                    "Time: %{x}<br>"
                    f"{parameter_column}: %{{y}}<extra></extra>"
                ),
                meta=[zone, station_name],
            )
        )

    figure.update_layout(
        title=f"{zone} - {parameter_column}",
        xaxis_title="Time",
        yaxis_title=parameter_column,
        hovermode="x unified",
        template="plotly_white",
        legend_title="Station",
        height=520,
    )
    figure.update_xaxes(rangeslider_visible=True)
    return figure


def render_zone_plot(zone: str, parameter_alias: str, year: int) -> None:
    zone_data = get_zone_parameter_frame(zone, parameter_alias, year)
    if zone_data.empty:
        display(HTML(f"<p><b>{zone}</b>: no data found for {year} and parameter <code>{parameter_alias}</code>.</p>"))
        return

    summary = (
        zone_data.groupby("station_name")[parameter_alias]
        .agg(["count", "min", "max", "mean"])
        .round(2)
        .rename_axis("station")
    )

    display(HTML(f"<h3>{zone}</h3>"))
    display(plot_weather_parameter(zone_data, parameter_alias, zone))
    display(summary)


def render_dashboard(zones: tuple[str, ...], parameter_alias: str, year: int) -> None:
    if not zones:
        display(HTML("<p>Select at least one zone.</p>"))
        return

    for zone in zones:
        render_zone_plot(zone, parameter_alias, year)


In [ ]:
available_zones = get_available_zones()
available_parameters = list(get_parameter_map())
available_years = get_available_years()

zone_selector = widgets.SelectMultiple(
    options=available_zones,
    value=(available_zones[0],),
    description="Zones",
    rows=min(8, len(available_zones)),
    layout=widgets.Layout(width="260px", height="180px"),
)

parameter_selector = widgets.Dropdown(
    options=[(get_display_name(parameter), parameter) for parameter in available_parameters],
    value=available_parameters[0],
    description="Parameter",
    layout=widgets.Layout(width="520px"),
)

year_selector = widgets.Dropdown(
    options=available_years,
    value=available_years[0],
    description="Year",
    layout=widgets.Layout(width="220px"),
)

controls = widgets.HBox([zone_selector, widgets.VBox([parameter_selector, year_selector])])
output = widgets.interactive_output(
    render_dashboard,
    {
        "zones": zone_selector,
        "parameter_alias": parameter_selector,
        "year": year_selector,
    },
)

display(controls, output)


Output()

In [5]:
# Example direct function calls without widgets:
zone_df = load_zone_weather("TPCODL")
filtered = filter_by_year(zone_df, 2017)
display(plot_weather_parameter(get_zone_parameter_frame("TPCODL", "temperature_2m", 2017), "temperature_2m", "TPCODL"))